# Graphs as Data & Message Passing

Companion notebook for the [Graphs as Data lesson](https://ml-viz-ruby.vercel.app/courses/graph-neural-networks/01-graphs-as-data).

We represent a small graph with an adjacency matrix and node features, implement one round of
**message passing** (mean-aggregate neighbors), watch information spread over multiple rounds, and
verify the whole thing is **permutation invariant**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1 — A graph as an adjacency matrix + features

Five nodes in a small graph. `A[u,v]=1` means an edge; we add self-loops so a node keeps its own
features during aggregation.

In [ ]:
# edges: 0-1, 1-2, 2-3, 3-4, 1-3  (undirected)
A = np.zeros((5, 5))
for u, v in [(0,1),(1,2),(2,3),(3,4),(1,3)]:
    A[u,v] = A[v,u] = 1
A = A + np.eye(5)                 # add self-loops

X = np.array([                    # one feature per node
    [1.0], [2.0], [3.0], [4.0], [5.0]
])
print('adjacency (with self-loops):\n', A)
print('node features:', X.ravel())

## 2 — One round of mean message passing

Each node's new feature is the **mean of its neighbors' features** (including itself). In matrix form
this is `D⁻¹ A X`, where `D` is the diagonal degree matrix — an elegant, permutation-respecting,
variable-neighbor-count operation.

In [ ]:
def mean_aggregate(A, X):
    deg = A.sum(axis=1, keepdims=True)     # number of neighbors (incl. self)
    return (A @ X) / deg                   # row-normalized neighbor mean

H1 = mean_aggregate(A, X)
print('after 1 round:', H1.ravel())
# node 0 (neighbors {0,1}): mean(1,2)=1.5 ; node 1 (neighbors {0,1,2,3}): mean(1,2,3,4)=2.5
assert np.isclose(H1[0,0], 1.5) and np.isclose(H1[1,0], 2.5)
print('\u2713 matches hand calculation')

## 3 — Information spreads with each round (receptive field)

After k rounds a node has mixed in information from up to k hops away. We track how node 0's value
evolves — and how, with many rounds, all nodes drift toward the same value (**over-smoothing**).

In [ ]:
H = X.copy()
for k in range(1, 8):
    H = mean_aggregate(A, H)
    print(f'round {k}: {H.ravel()}   spread={H.max()-H.min():.4f}')
print('\nNote how the spread shrinks toward 0 — that is over-smoothing.')

## 4 — Permutation invariance

Relabeling the nodes (permuting rows/cols of A and rows of X) must give the *same* result, just
reordered. We permute the graph, run message passing, and check it matches the un-permuted result
permuted the same way.

In [ ]:
perm = np.array([3, 0, 4, 1, 2])         # an arbitrary relabeling
P = np.eye(5)[perm]                       # permutation matrix

A_perm = P @ A @ P.T                       # relabel the graph
X_perm = P @ X
H_perm = mean_aggregate(A_perm, X_perm)

# message passing on the permuted graph == permuting the original result
assert np.allclose(H_perm, P @ mean_aggregate(A, X))
print('\u2713 message passing is permutation equivariant — node order does not matter')

## ✏️ Your turn

**Exercise.** Implement `aggregate(A, X, kind)` supporting `kind` in `{'sum', 'mean', 'max'}` —
the three permutation-invariant aggregators. For each node, pool the features of its neighbors
(the nonzero entries of that row of `A`, which already includes the self-loop).

In [ ]:
def aggregate(A, X, kind='mean'):
    out = np.zeros_like(X)
    for v in range(A.shape[0]):
        neighbors = np.where(A[v] > 0)[0]
        feats = X[neighbors]
        # TODO(you): set out[v] to the sum / mean / max over `feats` according to `kind`
        ...
    return out

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert np.allclose(aggregate(A, X, 'mean'), mean_aggregate(A, X))
assert np.isclose(aggregate(A, X, 'sum')[0, 0], 3.0)     # node 0: 1+2
assert np.isclose(aggregate(A, X, 'max')[1, 0], 4.0)     # node 1 neighbors max = 4
# permutation invariance holds for any aggregator
for k in ['sum', 'mean', 'max']:
    assert np.allclose(aggregate(P @ A @ P.T, P @ X, k), P @ aggregate(A, X, k))
print('\u2713 all three aggregators are correct and permutation equivariant')

<details>
<summary>Solution</summary>

```python
def aggregate(A, X, kind='mean'):
    out = np.zeros_like(X)
    for v in range(A.shape[0]):
        feats = X[np.where(A[v] > 0)[0]]
        if kind == 'sum':
            out[v] = feats.sum(axis=0)
        elif kind == 'mean':
            out[v] = feats.mean(axis=0)
        elif kind == 'max':
            out[v] = feats.max(axis=0)
    return out
```

Sum, mean, and max are all permutation invariant — the order of the neighbor set doesn't change the
result. That invariance is precisely what makes them valid graph aggregators.

</details>